# HR Employee Attrition Analysis

Explore workforce factors associated with employee attrition using synthetic HR data.

**Important:** relationships are descriptive and do not establish causation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(99)
n = 5000
departments = ['Engineering', 'Sales', 'HR', 'Marketing', 'Finance', 'Operations']

df = pd.DataFrame({
    'EmployeeID': [f'EMP{i:04d}' for i in range(1, n + 1)],
    'Age': rng.integers(22, 60, n),
    'Department': rng.choice(departments, n),
    'MonthlyIncome': np.round(rng.uniform(3000, 20000, n), 2),
    'YearsAtCompany': rng.integers(0, 20, n),
    'JobSatisfaction': rng.integers(1, 5, n),
    'WorkLifeBalance': rng.integers(1, 5, n),
    'OverTime': rng.integers(0, 2, n),
    'YearsSinceLastPromotion': rng.integers(0, 10, n),
    'DistanceFromHome': rng.integers(1, 50, n),
})

score = (
    -0.30 * df['JobSatisfaction']
    -0.20 * df['WorkLifeBalance']
    + 0.40 * df['OverTime']
    - 0.15 * np.log1p(df['YearsAtCompany'])
    + 0.05 * df['YearsSinceLastPromotion']
    + 0.15 * df['DistanceFromHome'] / 50
    + rng.normal(0, 0.3, n)
)
prob = 1 / (1 + np.exp(-score))
df['Attrition'] = (rng.random(n) < prob).astype(int)

df.head()


In [ ]:
summary = pd.Series({
    'Employees': len(df),
    'Attrition Rate (%)': round(df['Attrition'].mean() * 100, 2),
    'Average Monthly Income': round(df['MonthlyIncome'].mean(), 2),
    'Average Tenure (years)': round(df['YearsAtCompany'].mean(), 2),
    'Overtime Share (%)': round(df['OverTime'].mean() * 100, 2),
})
summary


In [ ]:
department_attrition = (
    df.groupby('Department')['Attrition']
      .agg(Employees='count', Attrited='sum', AttritionRate='mean')
)
department_attrition['AttritionRate'] = (department_attrition['AttritionRate'] * 100).round(2)
department_attrition.sort_values('AttritionRate', ascending=False)


In [ ]:
numeric = df.select_dtypes('number')
correlations = numeric.corr()['Attrition'].drop('Attrition').sort_values(key=abs, ascending=False)
correlations


In [ ]:
correlations.sort_values().plot.barh(figsize=(8, 4))
plt.title('Correlation with Employee Attrition')
plt.xlabel('Pearson correlation')
plt.tight_layout()
plt.show()
